In [41]:
import os
import pandas as pd
from PIL import Image
from pathlib import Path

In [42]:
MODEL_DIR = Path("model")
(MODEL_DIR).mkdir(parents=False, exist_ok=True)
CURRENT_DIR = Path.cwd()

# If 'Datasets' is 3 levels up from your notebook:
PROJECT_ROOT = CURRENT_DIR.parents[2]  # Adjust index (0, 1, 2...) if needed

DATASET_DIR = (
    PROJECT_ROOT
    / "Datasets"
    / "Brain Tumor 12K Mri Images"
    / "Images_"
    / "Images_"
    / "Embryonic Tumors"
)


NEGATIVE_CLASS = "Normal"
POSITIVE_CLASS = "Embryonic Tumors"

IMG_SIZE = 224
SEG_SIZE = 256
BATCH_SIZE = 16
SEG_BATCH_SIZE = 8
VAL_SIZE = 0.15
TEST_SIZE = 0.15

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_TOKENS = ("mask", "segmentation", "_seg", "label", "consensus")

print("Dataset:", DATASET_DIR.resolve())
print("Exists:", DATASET_DIR.exists())

Dataset: C:\Users\sinco\Documents\GitHub\Medical-Imaging-Research\Datasets\Brain Tumor 12K Mri Images\Images_\Images_\Embryonic Tumors
Exists: True


In [43]:
def is_mask_file(path):
    stem = Path(path).stem.lower()
    return any(tok in stem for tok in MASK_TOKENS)

def infer_group_id(path):
    stem = Path(path).stem.lower()
    stem = re.sub(r"(?:_?mask(?:_consensus)?|_?seg(?:mentation)?|_?label)$", "", stem)
    stem = re.sub(r"[_\- ]?(slice|image|img|scan)[_\- ]?\d+$", "", stem)
    stem = re.sub(r"[_\- ]?\d{1,4}$", "", stem)
    return stem.strip("_- ") or Path(path).stem.lower()

def build_classification_dataframe(dataset_dir):
    mapping = {NEGATIVE_CLASS: 0, POSITIVE_CLASS: 1}
    rows = []
    for class_name, label in mapping.items():
        class_dir = dataset_dir / class_name
        if not class_dir.exists():
            print("WARNING: missing:", class_dir)
            continue
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS:
                rows.append({
                    "path": str(p),
                    "class_name": class_name,
                    "label": label,
                    "is_mask": is_mask_file(p),
                    "group_id": infer_group_id(p),
                })
    return pd.DataFrame(rows)

df_all = build_classification_dataframe(DATASET_DIR)
if df_all.empty:
    raise FileNotFoundError("No images found. Check DATASET_DIR and class folder names.")

print("All discovered files:", len(df_all))
display(df_all.groupby(["class_name", "is_mask"]).size().rename("count"))

df = df_all.loc[~df_all.is_mask].copy().reset_index(drop=True)
print("Classification images after mask exclusion:", len(df))
display(df.head())

FileNotFoundError: No images found. Check DATASET_DIR and class folder names.